# 19.3 赌博机 vs A/B 测试 / Bandits vs A/B Testing

**中文**：我们在 15.10、17.10 学过多臂赌博机的算法。本节回答一个**实战决策问题**:*"要优化一个东西,我该用固定的 A/B 测试,还是用自适应的赌博机?"* 这是数据科学/产品面试的高频题。答案的核心是一个根本权衡——**"学习(learn)" vs "赚取(earn)"**。
**English**: We covered multi-armed bandit algorithms in 15.10 and 17.10. This section answers a **practical decision**: *"to optimize something, should I use a fixed A/B test or an adaptive bandit?"* A frequent data-science/product interview question. The answer centers on a fundamental trade-off — **"learn" vs "earn."**

---

**中文**：
- **A/B 测试**:**固定**分流(如永远 50/50),跑满预定样本量,再**一次性**决定赢家。它把整个实验期都用于"学习"——**代价**是:实验期间,一半用户一直看到较差的版本(这部分损失叫**遗憾, regret**)。**好处**:每个版本都拿到**无偏、可信区间清晰**的估计,统计干净,适合需要精确结论的**一次性重大决策**。
- **赌博机(bandit)**:**自适应**分流——边跑边把更多流量倒向当前更好的版本(如 Thompson 采样)。它**边学边赚**,最小化遗憾。**好处**:实验期间累计收益更高,适合**短命内容**(新闻、广告、推送)和**持续优化**。**代价**:对较差版本采样越来越少,其估计**有偏、方差大**,难做严谨的统计推断。

**English**:
- **A/B test**: **fixed** allocation (e.g. always 50/50), run to a preset sample size, then decide the winner **once**. The whole experiment is spent "learning" — the **cost**: throughout, half the users keep seeing the worse version (this loss is called **regret**). The **benefit**: every version gets an **unbiased estimate with clean confidence intervals**, statistically clean, ideal for a **one-time major decision** needing precise conclusions.
- **Bandit**: **adaptive** allocation — shift more traffic to the currently-better version as you go (e.g. Thompson sampling). It **earns while learning**, minimizing regret. The **benefit**: higher cumulative reward during the experiment, ideal for **short-lived content** (news, ads, notifications) and **continuous optimization**. The **cost**: the worse version is sampled less and less, so its estimate is **biased, high-variance**, hard for rigorous statistical inference.

**中文**：一句话总结这个权衡:**A/B 是"先充分学习,再做决定";赌博机是"边赚钱边学习"**。哪个好没有绝对答案,取决于你更在乎"实验期间少损失"还是"拿到每个版本的精确评估"。
**English**: In one line: **A/B is "learn fully, then decide"; a bandit is "earn while learning."** Neither is universally better; it depends on whether you care more about "losing less during the experiment" or "getting precise evaluations of every version."

> 💡 **面试速查 / Interview cheat-sheet（★★★ 产品决策必考）**
> **中文**：**A/B vs 赌博机 = learn vs earn**。**A/B**:固定分流→每臂无偏估计+干净CI, 但实验期遗憾大; 适合**重大一次性决策、需精确效应/多指标/长期效应、少量版本**。**赌博机**:自适应分流(Thompson/UCB)→遗憾小(边赚边学), 但劣臂估计有偏、推断难; 适合**短命内容(新闻/广告/推送)、很多版本、持续优化、遗憾成本高**。**关键坑**:①赌博机假设**平稳**(奖励分布不随时间变), 非平稳要用衰减/滑窗;②赌博机的**统计推断难**(自适应采样破坏独立性→需专门方法);③想两全其美→**Best-arm identification / 固定预算赌博机 / 上下文赌博机**。默认:要严谨因果结论用 A/B, 要在线收益最大化用赌博机。
> **English**: **A/B vs bandit = learn vs earn**. **A/B**: fixed allocation → unbiased per-arm estimates + clean CIs, but high regret during the test; good for **major one-time decisions, needing precise effects / many metrics / long-term effects, few variants**. **Bandit**: adaptive allocation (Thompson/UCB) → low regret (earn while learning), but biased worse-arm estimates and hard inference; good for **short-lived content (news/ads/push), many variants, continuous optimization, high regret cost**. **Key pitfalls**: ① bandits assume **stationarity** (reward distribution constant over time); non-stationary needs decay/sliding windows; ② bandits' **statistical inference is hard** (adaptive sampling breaks independence → needs special methods); ③ to get both worlds → **best-arm identification / fixed-budget bandits / contextual bandits**. Default: use A/B for rigorous causal conclusions, a bandit for online reward maximization.


In [ ]:

# ============================================================
# 模拟:同一个两版本场景, A/B(固定) vs Thompson(自适应)/ A/B (fixed) vs Thompson (adaptive)
# 中文:两个版本真实点击率 10% vs 13%(B更好但你不知道)。跑 T 轮, 每轮给一个用户展示一个版本。
#      目标对比:①累计遗憾(展示较差版本的损失);②对每个版本点击率的估计质量。
# English: two variants with true CTR 10% vs 13% (B better, unknown to you). T rounds, each shows a user one variant.
#      Compare: ① cumulative regret; ② estimation quality of each variant's CTR.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(0)
p_true=[0.10, 0.13]; T=20000; best=max(p_true)

def run_ab(explore=4000, trials=120):
    reg=np.zeros(T); alloc=np.zeros(T); est=[]
    for _ in range(trials):
        n=[0,0]; s=[0,0]; cum=0
        for t in range(T):
            a = t%2 if t<explore else int(np.argmax([s[0]/max(n[0],1), s[1]/max(n[1],1)]))  # 前期50/50后期贪心
            r = rng.random()<p_true[a]; n[a]+=1; s[a]+=r
            cum += best-p_true[a]; reg[t]+=cum; alloc[t]+=(a==1)
        est.append([s[0]/n[0], s[1]/n[1]])
    return reg/trials, alloc/trials, np.array(est)

def run_thompson(trials=120):
    reg=np.zeros(T); alloc=np.zeros(T); est=[]
    for _ in range(trials):
        al=[1,1]; be=[1,1]; n=[0,0]; s=[0,0]; cum=0
        for t in range(T):
            a=int(np.argmax([rng.beta(al[0],be[0]), rng.beta(al[1],be[1])]))   # 从后验采样选臂 / Thompson
            r=rng.random()<p_true[a]; al[a]+=r; be[a]+=1-r; n[a]+=1; s[a]+=r
            cum += best-p_true[a]; reg[t]+=cum; alloc[t]+=(a==1)
        est.append([s[0]/max(n[0],1), s[1]/max(n[1],1)])
    return reg/trials, alloc/trials, np.array(est)

ab_reg, ab_alloc, ab_est = run_ab()
ts_reg, ts_alloc, ts_est = run_thompson()
print(f"真实点击率 / true CTR: A={p_true[0]:.0%}, B={p_true[1]:.0%}")
print(f"累计遗憾(越低越好)/ cumulative regret: A/B={ab_reg[-1]:.0f}, Thompson={ts_reg[-1]:.0f}")
print(f"→ Thompson 遗憾少 {(1-ts_reg[-1]/ab_reg[-1])*100:.0f}% (边学边赚) / bandit earns while learning")


**中文**：Thompson 赌博机的累计遗憾明显低于 A/B——它没有"傻等",而是很快把流量倒向更好的 B。但**天下没有免费午餐**:看两种方法对**较差版本 A** 的估计质量。A/B 全程给 A 稳定的一半流量,估计精准;赌博机很快抛弃 A,对 A 的估计**样本少、有偏**。这正是"赚取"的代价——牺牲了对劣势版本的了解。
**English**: Thompson's cumulative regret is clearly lower than A/B's — it doesn't "wait dumbly" but quickly shifts traffic to the better B. But **there's no free lunch**: look at each method's estimate of the **worse variant A**. A/B gives A a steady half of traffic throughout, so its estimate is precise; the bandit abandons A quickly, so its estimate of A is **low-sample and biased**. That is the cost of "earning" — sacrificing knowledge of the losing variant.


In [ ]:

# ============================================================
# 估计质量对比 + 可视化 / estimation quality + visualization
# ============================================================
print(f"{'':<12}{'对A的估计(真10%)':>18}{'对B的估计(真13%)':>18}")
print(f"{'A/B':<12}{ab_est[:,0].mean():>17.1%}{ab_est[:,1].mean():>18.1%}  (两个都准)")
print(f"{'Thompson':<12}{ts_est[:,0].mean():>17.1%}{ts_est[:,1].mean():>18.1%}  (A 有偏/方差大)")
print(f"对A估计的标准差 / std of A-estimate: A/B={ab_est[:,0].std():.3f} vs Thompson={ts_est[:,0].std():.3f}")

fig,ax=plt.subplots(1,3,figsize=(17,4.6))
# ① 累计遗憾 / cumulative regret
ax[0].plot(ab_reg,color="#C44E52",label=f"A/B 固定 (遗憾 {ab_reg[-1]:.0f})")
ax[0].plot(ts_reg,color="#4C72B0",label=f"Thompson 自适应 (遗憾 {ts_reg[-1]:.0f})")
ax[0].set_title("累计遗憾:赌博机边学边赚 / cumulative regret"); ax[0].set_xlabel("用户 t"); ax[0].set_ylabel("累计遗憾"); ax[0].legend(fontsize=8)
# ② 流量分配 / traffic allocation to B over time
ax[1].plot(ab_alloc,color="#C44E52",label="A/B(固定~50%后跳变)")
ax[1].plot(ts_alloc,color="#4C72B0",label="Thompson(渐倒向B)")
ax[1].axhline(0.5,ls=":",color="gray"); ax[1].set_title("给较好版本B的流量比例 / traffic to B"); ax[1].set_xlabel("用户 t"); ax[1].set_ylabel("分给B的比例"); ax[1].legend(fontsize=8); ax[1].set_ylim(0,1.05)
# ③ 估计质量:对劣势版本A的估计分布 / estimate of the losing arm A
ax[2].hist(ab_est[:,0],bins=25,alpha=0.6,color="#C44E52",label="A/B 对A的估计")
ax[2].hist(ts_est[:,0],bins=25,alpha=0.6,color="#4C72B0",label="Thompson 对A的估计")
ax[2].axvline(p_true[0],color="g",lw=2,label="真值 10%")
ax[2].set_title("劣势版本A的估计:A/B更准 / estimate of losing arm A"); ax[2].set_xlabel("对A点击率的估计"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/ci03_viz.png",dpi=80); plt.show()
print("A/B 对两个版本都给出精准估计; 赌博机赚得多但对劣势版本'了解不足'")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **learn vs earn 的权衡是真实且对称的**:Thompson 赌博机的累计遗憾比 A/B 低一半多(它很快把流量倒向更好的 B,少损失)。**但代价清清楚楚**:它对劣势版本 A 的估计**又偏又不稳**(右图蓝色分布散、偏离真值),因为它很快就不怎么给 A 流量了。A/B 虽然遗憾高,却对**两个版本都给出精准无偏的估计**——这在"你需要知道每个版本到底多好"(如汇报、长期决策、多指标权衡)时至关重要。
2. **选择取决于你的目标,不是"谁更先进"**:
   - **用 A/B**:重大一次性决策、需要精确效应量和置信区间、要评估多个指标(含护栏)、关心长期效应、版本不多。
   - **用赌博机**:内容短命(今天的头条明天就过时,没时间慢慢 A/B)、版本很多、目标就是最大化在线收益、探索成本高。
3. **诚实的陷阱**:①赌博机默认**环境平稳**——如果版本的真实效果随时间变(promotion 结束、用户审美疲劳),朴素赌博机会被旧数据误导,要用**衰减/滑动窗口**;②赌博机的**统计显著性很难算**(自适应采样让样本不再独立同分布,普通 t 检验会失效),想要严谨结论得用专门的 anytime-valid 推断;③现实里常用**混合方案**:先短暂均匀探索(拿到干净估计)再切赌博机,或用**上下文赌博机**(17.10)兼顾个性化。别把"赌博机更酷"当成默认选择。

**English**:
1. **The learn-vs-earn trade-off is real and symmetric**: Thompson's cumulative regret is less than half of A/B's (it quickly shifts traffic to the better B, losing less). **But the cost is clear**: its estimate of the losing variant A is **biased and unstable** (blue distribution spread and off-target, right plot), because it soon stops giving A traffic. A/B, though higher-regret, gives **precise unbiased estimates of both variants** — crucial when "you need to know exactly how good each variant is" (reporting, long-term decisions, multi-metric trade-offs).
2. **The choice depends on your goal, not "which is more advanced"**:
   - **Use A/B**: major one-time decisions, need precise effect sizes and CIs, evaluate multiple metrics (incl. guardrails), care about long-term effects, few variants.
   - **Use a bandit**: short-lived content (today's headline is stale tomorrow, no time to slow-A/B), many variants, the goal is maximizing online reward, high exploration cost.
3. **Honest pitfalls**: ① bandits assume a **stationary** environment — if a variant's true effect changes over time (a promotion ends, users tire), a naive bandit is misled by stale data and needs **decay / sliding windows**; ② a bandit's **statistical significance is hard** (adaptive sampling breaks i.i.d., so a plain t-test fails); rigorous conclusions need specialized anytime-valid inference; ③ in practice use **hybrids**: a brief uniform exploration (for clean estimates) then switch to a bandit, or a **contextual bandit** (17.10) for personalization. Don't default to "bandits are cooler."

> 💼 **实战视角 / Practical angle**
> **中文**:决策框架:①**要因果结论/汇报/长期决策 → A/B**(干净、可推断);②**要在线收益、内容短命、版本多 → 赌博机**(少遗憾)。工业实践:新闻/广告/推送标题优化用赌博机(Yahoo/头条), 产品功能大改用 A/B(需严谨评估)。**折中方案**:epsilon-A/B(留一小部分固定流量给所有臂保证可推断)、Best-arm identification(以最快锁定最优为目标)、上下文赌博机(个性化)。面试金句:*"A/B 是 learn-then-decide、每臂无偏但实验期遗憾大; 赌博机是 earn-while-learning、遗憾小但劣臂估计有偏、推断难; 重大决策/需精确效应用 A/B, 短命内容/在线收益最大化用赌博机——还要注意赌博机的平稳性假设与推断困难。"*
> **English**: Decision framework: ① **need causal conclusions / reporting / long-term decisions → A/B** (clean, inferable); ② **need online reward, short-lived content, many variants → bandit** (low regret). Industry: news/ads/push headline optimization uses bandits (Yahoo/Toutiao); major product changes use A/B (rigorous evaluation). **Compromises**: epsilon-A/B (reserve a small fixed slice for all arms to keep inference valid), best-arm identification (goal = lock onto the best fastest), contextual bandits (personalization). Interview line: *"A/B is learn-then-decide — unbiased per arm but high in-experiment regret; a bandit is earn-while-learning — low regret but biased worse-arm estimates and hard inference; use A/B for major decisions / precise effects, a bandit for short-lived content / online reward maximization — and watch the bandit's stationarity assumption and inference difficulty."*

---
### 小结 / Summary
- **中文**:A/B(固定分流)=learn-then-decide, 每臂无偏+干净CI, 但实验期遗憾大。
- **English**: A/B (fixed allocation) = learn-then-decide, unbiased per arm + clean CIs, but high in-experiment regret.
- **中文**:赌博机(自适应)=earn-while-learning, 遗憾小, 但劣臂估计有偏、统计推断难、假设平稳。
- **English**: Bandit (adaptive) = earn-while-learning, low regret, but biased worse-arm estimates, hard inference, assumes stationarity.
- **中文**:选择看目标:严谨因果决策用 A/B, 短命内容/在线收益最大化用赌博机; 常用混合方案。
- **English**: Choose by goal: A/B for rigorous causal decisions, bandit for short-lived content / online reward; hybrids are common.
